# 03 Train and Evaluate

Train the multi-task model with alternating batches and evaluate both ABSA and emotion tasks.

## Workflow

- Load processed data and local BERT model
- Rebuild datasets and dataloaders
- Define multi-task model
- Alternate ABSA and emotion batches during training
- Evaluate on dev and test splits
- Save the trained checkpoint

In [1]:
from pathlib import Path
import json
import math
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
from torch.optim import AdamW
from torch.utils.data import Dataset, DataLoader
from transformers import BertTokenizer, BertModel, get_linear_schedule_with_warmup
from sklearn.metrics import accuracy_score, f1_score, classification_report

pd.set_option('display.max_colwidth', 120)

SEED = 42
torch.manual_seed(SEED)
np.random.seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
PROJECT_DIR = Path.cwd()
PROCESSED_DIR = PROJECT_DIR / 'data' / 'processed'
MODEL_DIR = PROJECT_DIR / 'models'
CHECKPOINT_DIR = PROJECT_DIR / 'checkpoints'
CHECKPOINT_DIR.mkdir(parents=True, exist_ok=True)

DEVICE

d:\important\Conda_Envs\base_D\Lib\site-packages\keras\src\export\tf2onnx_lib.py:8: FutureWarning: In the future `np.object` will be defined as the corresponding NumPy scalar.
  if not hasattr(np, "object"):


device(type='cuda')

In [2]:
absa_df = pd.read_csv(PROCESSED_DIR / 'absa_semeval_2014.csv')
emotion_df = pd.read_csv(PROCESSED_DIR / 'emotion_goemotions_6class.csv')

with open(PROCESSED_DIR / 'label_metadata.json', 'r', encoding='utf-8') as f:
    label_metadata = json.load(f)

ABSA_LABEL2ID = label_metadata['absa_label2id']
EMOTION_LABEL2ID = label_metadata['emotion_label2id']
ID2ABSA_LABEL = {int(v): k for k, v in ABSA_LABEL2ID.items()}
ID2EMOTION_LABEL = {int(v): k for k, v in EMOTION_LABEL2ID.items()}

absa_train = absa_df[absa_df['split'] == 'train'].reset_index(drop=True)
absa_dev = absa_df[absa_df['split'] == 'dev'].reset_index(drop=True)
absa_test = absa_df[absa_df['split'] == 'test'].reset_index(drop=True)

emotion_train = emotion_df[emotion_df['split'] == 'train'].reset_index(drop=True)
emotion_dev = emotion_df[emotion_df['split'] == 'dev'].reset_index(drop=True)
emotion_test = emotion_df[emotion_df['split'] == 'test'].reset_index(drop=True)

print('ABSA train/dev/test:', len(absa_train), len(absa_dev), len(absa_test))
print('Emotion train/dev/test:', len(emotion_train), len(emotion_dev), len(emotion_test))

ABSA train/dev/test: 4840 605 606
Emotion train/dev/test: 39555 4946 4968


In [3]:
MODEL_NAME = str(MODEL_DIR)
MAX_LENGTH = 128
BATCH_SIZE = 16
NUM_EPOCHS = 3
LEARNING_RATE = 2e-5

tokenizer = BertTokenizer.from_pretrained(MODEL_NAME, local_files_only=True)
tokenizer

BertTokenizer(name_or_path='d:\15_MAI\pynotebook\nlpProject\models', vocab_size=30522, model_max_length=512, is_fast=False, padding_side='right', truncation_side='right', special_tokens={'unk_token': '[UNK]', 'sep_token': '[SEP]', 'pad_token': '[PAD]', 'cls_token': '[CLS]', 'mask_token': '[MASK]'}, clean_up_tokenization_spaces=True, added_tokens_decoder={
	0: AddedToken("[PAD]", rstrip=False, lstrip=False, single_word=False, normalized=False, special=True),
	100: AddedToken("[UNK]", rstrip=False, lstrip=False, single_word=False, normalized=False, special=True),
	101: AddedToken("[CLS]", rstrip=False, lstrip=False, single_word=False, normalized=False, special=True),
	102: AddedToken("[SEP]", rstrip=False, lstrip=False, single_word=False, normalized=False, special=True),
	103: AddedToken("[MASK]", rstrip=False, lstrip=False, single_word=False, normalized=False, special=True),
}
)

## Dataset and Loader

In [4]:
class ABSADataset(Dataset):
    def __init__(self, dataframe: pd.DataFrame, tokenizer, max_length: int = 128):
        self.df = dataframe.reset_index(drop=True)
        self.tokenizer = tokenizer
        self.max_length = max_length

    def __len__(self):
        return len(self.df)

    def __getitem__(self, idx):
        row = self.df.iloc[idx]
        encoded = self.tokenizer(
            text=str(row['text']),
            text_pair=str(row['aspect']),
            padding='max_length',
            truncation=True,
            max_length=self.max_length,
            return_tensors='pt'
        )

        item = {
            'input_ids': encoded['input_ids'].squeeze(0),
            'attention_mask': encoded['attention_mask'].squeeze(0),
            'labels': torch.tensor(int(row['label_id']), dtype=torch.long),
        }
        if 'token_type_ids' in encoded:
            item['token_type_ids'] = encoded['token_type_ids'].squeeze(0)
        return item

class EmotionDataset(Dataset):
    def __init__(self, dataframe: pd.DataFrame, tokenizer, max_length: int = 128):
        self.df = dataframe.reset_index(drop=True)
        self.tokenizer = tokenizer
        self.max_length = max_length

    def __len__(self):
        return len(self.df)

    def __getitem__(self, idx):
        row = self.df.iloc[idx]
        encoded = self.tokenizer(
            text=str(row['text']),
            padding='max_length',
            truncation=True,
            max_length=self.max_length,
            return_tensors='pt'
        )

        item = {
            'input_ids': encoded['input_ids'].squeeze(0),
            'attention_mask': encoded['attention_mask'].squeeze(0),
            'labels': torch.tensor(int(row['label_id']), dtype=torch.long),
        }
        if 'token_type_ids' in encoded:
            item['token_type_ids'] = encoded['token_type_ids'].squeeze(0)
        return item

In [5]:
absa_train_loader = DataLoader(ABSADataset(absa_train, tokenizer, MAX_LENGTH), batch_size=BATCH_SIZE, shuffle=True)
absa_dev_loader = DataLoader(ABSADataset(absa_dev, tokenizer, MAX_LENGTH), batch_size=BATCH_SIZE, shuffle=False)
absa_test_loader = DataLoader(ABSADataset(absa_test, tokenizer, MAX_LENGTH), batch_size=BATCH_SIZE, shuffle=False)

emotion_train_loader = DataLoader(EmotionDataset(emotion_train, tokenizer, MAX_LENGTH), batch_size=BATCH_SIZE, shuffle=True)
emotion_dev_loader = DataLoader(EmotionDataset(emotion_dev, tokenizer, MAX_LENGTH), batch_size=BATCH_SIZE, shuffle=False)
emotion_test_loader = DataLoader(EmotionDataset(emotion_test, tokenizer, MAX_LENGTH), batch_size=BATCH_SIZE, shuffle=False)

len(absa_train_loader), len(emotion_train_loader)

(303, 2473)

## Multi-task Model

In [6]:
class MultiTaskBERT(nn.Module):
    def __init__(self, model_name, absa_num_labels=4, emotion_num_labels=6, dropout=0.1):
        super().__init__()
        self.encoder = BertModel.from_pretrained(model_name, local_files_only=True)
        hidden_size = self.encoder.config.hidden_size

        self.dropout = nn.Dropout(dropout)
        self.absa_classifier = nn.Linear(hidden_size, absa_num_labels)
        self.emotion_classifier = nn.Linear(hidden_size, emotion_num_labels)

    def forward(self, input_ids, attention_mask, token_type_ids=None, task='absa'):
        outputs = self.encoder(
            input_ids=input_ids,
            attention_mask=attention_mask,
            token_type_ids=token_type_ids
        )

        cls_output = outputs.last_hidden_state[:, 0]
        cls_output = self.dropout(cls_output)

        if task == 'absa':
            return self.absa_classifier(cls_output)
        if task == 'emotion':
            return self.emotion_classifier(cls_output)
        raise ValueError("task must be 'absa' or 'emotion'")

In [7]:
model = MultiTaskBERT(
    model_name=MODEL_NAME,
    absa_num_labels=len(ABSA_LABEL2ID),
    emotion_num_labels=len(EMOTION_LABEL2ID)
).to(DEVICE)

absa_loss_fn = nn.CrossEntropyLoss()
emotion_loss_fn = nn.CrossEntropyLoss()

total_steps = max(len(absa_train_loader), len(emotion_train_loader)) * NUM_EPOCHS
optimizer = AdamW(model.parameters(), lr=LEARNING_RATE)
scheduler = get_linear_schedule_with_warmup(
    optimizer,
    num_warmup_steps=int(0.1 * total_steps),
    num_training_steps=total_steps
)

total_steps

7419

## Helper Functions

In [8]:
def move_batch_to_device(batch, device):
    moved = {}
    for key, value in batch.items():
        if isinstance(value, torch.Tensor):
            moved[key] = value.to(device)
        else:
            moved[key] = value
    return moved

def evaluate_model(model, data_loader, task, loss_fn, id2label):
    model.eval()
    losses = []
    all_preds = []
    all_labels = []

    with torch.no_grad():
        for batch in data_loader:
            batch = move_batch_to_device(batch, DEVICE)
            logits = model(
                input_ids=batch['input_ids'],
                attention_mask=batch['attention_mask'],
                token_type_ids=batch.get('token_type_ids'),
                task=task
            )

            loss = loss_fn(logits, batch['labels'])
            preds = torch.argmax(logits, dim=1)

            losses.append(loss.item())
            all_preds.extend(preds.cpu().numpy().tolist())
            all_labels.extend(batch['labels'].cpu().numpy().tolist())

    metrics = {
        'loss': float(np.mean(losses)),
        'accuracy': accuracy_score(all_labels, all_preds),
        'macro_f1': f1_score(all_labels, all_preds, average='macro'),
        'report': classification_report(
            all_labels,
            all_preds,
            target_names=[id2label[i] for i in sorted(id2label.keys())],
            digits=4
        )
    }
    return metrics

## Training

In [9]:
import gc
gc.collect()
torch.cuda.empty_cache()

In [10]:
best_score = -1
history = []

for epoch in range(NUM_EPOCHS):
    model.train()
    absa_iter = iter(absa_train_loader)
    emotion_iter = iter(emotion_train_loader)

    num_steps = max(len(absa_train_loader), len(emotion_train_loader))
    running_absa_loss = []
    running_emotion_loss = []

    for step in range(num_steps):
        try:
            absa_batch = next(absa_iter)
        except StopIteration:
            absa_iter = iter(absa_train_loader)
            absa_batch = next(absa_iter)

        absa_batch = move_batch_to_device(absa_batch, DEVICE)
        optimizer.zero_grad()
        absa_logits = model(
            input_ids=absa_batch['input_ids'],
            attention_mask=absa_batch['attention_mask'],
            token_type_ids=absa_batch.get('token_type_ids'),
            task='absa'
        )
        absa_loss = absa_loss_fn(absa_logits, absa_batch['labels'])
        absa_loss.backward()
        optimizer.step()
        scheduler.step()
        running_absa_loss.append(absa_loss.item())

        try:
            emotion_batch = next(emotion_iter)
        except StopIteration:
            emotion_iter = iter(emotion_train_loader)
            emotion_batch = next(emotion_iter)

        emotion_batch = move_batch_to_device(emotion_batch, DEVICE)
        optimizer.zero_grad()
        emotion_logits = model(
            input_ids=emotion_batch['input_ids'],
            attention_mask=emotion_batch['attention_mask'],
            token_type_ids=emotion_batch.get('token_type_ids'),
            task='emotion'
        )
        emotion_loss = emotion_loss_fn(emotion_logits, emotion_batch['labels'])
        emotion_loss.backward()
        optimizer.step()
        scheduler.step()
        running_emotion_loss.append(emotion_loss.item())

    absa_dev_metrics = evaluate_model(model, absa_dev_loader, 'absa', absa_loss_fn, ID2ABSA_LABEL)
    emotion_dev_metrics = evaluate_model(model, emotion_dev_loader, 'emotion', emotion_loss_fn, ID2EMOTION_LABEL)

    avg_train_loss = float(np.mean(running_absa_loss + running_emotion_loss))
    combined_score = (absa_dev_metrics['macro_f1'] + emotion_dev_metrics['macro_f1']) / 2

    history.append({
        'epoch': epoch + 1,
        'train_loss': avg_train_loss,
        'absa_dev_acc': absa_dev_metrics['accuracy'],
        'absa_dev_macro_f1': absa_dev_metrics['macro_f1'],
        'emotion_dev_acc': emotion_dev_metrics['accuracy'],
        'emotion_dev_macro_f1': emotion_dev_metrics['macro_f1'],
        'combined_score': combined_score,
    })

    print(f"Epoch {epoch + 1}/{NUM_EPOCHS}")
    print(f"Train loss: {avg_train_loss:.4f}")
    print(f"ABSA dev acc: {absa_dev_metrics['accuracy']:.4f} | macro-F1: {absa_dev_metrics['macro_f1']:.4f}")
    print(f"Emotion dev acc: {emotion_dev_metrics['accuracy']:.4f} | macro-F1: {emotion_dev_metrics['macro_f1']:.4f}")
    print('-' * 60)

    if combined_score > best_score:
        best_score = combined_score
        torch.save(model.state_dict(), CHECKPOINT_DIR / 'best_multitask_bert.pt')

history_df = pd.DataFrame(history)
history_df

Epoch 1/3
Train loss: 0.5775
ABSA dev acc: 0.8231 | macro-F1: 0.6367
Emotion dev acc: 0.7188 | macro-F1: 0.6668
------------------------------------------------------------
Epoch 2/3
Train loss: 0.3116
ABSA dev acc: 0.8298 | macro-F1: 0.6458
Emotion dev acc: 0.7184 | macro-F1: 0.6753
------------------------------------------------------------
Epoch 3/3
Train loss: 0.2887
ABSA dev acc: 0.8298 | macro-F1: 0.6458
Emotion dev acc: 0.7184 | macro-F1: 0.6753
------------------------------------------------------------


,epoch,train_loss,absa_dev_acc,absa_dev_macro_f1,emotion_dev_acc,emotion_dev_macro_f1,combined_score
0,1,0.577478,0.823140,0.636732,0.718763,0.666814,0.651773
1,2,0.311620,0.829752,0.645834,0.718358,0.675326,0.660580
2,3,0.288709,0.829752,0.645834,0.718358,0.675326,0.660580


## Load Best Checkpoint

In [11]:
model.load_state_dict(torch.load(CHECKPOINT_DIR / 'best_multitask_bert.pt', map_location=DEVICE))
model.eval()
print('Best checkpoint loaded.')

Best checkpoint loaded.


## Test Evaluation

In [12]:
absa_test_metrics = evaluate_model(model, absa_test_loader, 'absa', absa_loss_fn, ID2ABSA_LABEL)
emotion_test_metrics = evaluate_model(model, emotion_test_loader, 'emotion', emotion_loss_fn, ID2EMOTION_LABEL)

print('ABSA Test Metrics')
print(absa_test_metrics['report'])
print('Emotion Test Metrics')
print(emotion_test_metrics['report'])

ABSA Test Metrics
              precision    recall  f1-score   support

    positive     0.8927    0.8956    0.8942       316
    negative     0.7738    0.7784    0.7761       167
     neutral     0.6577    0.6697    0.6636       109
    conflict     0.4000    0.2857    0.3333        14

    accuracy                         0.8086       606
   macro avg     0.6811    0.6574    0.6668       606
weighted avg     0.8063    0.8086    0.8072       606

Emotion Test Metrics
              precision    recall  f1-score   support

         joy     0.8153    0.8578    0.8360      1863
       anger     0.6009    0.5880    0.5944       648
     sadness     0.6958    0.5901    0.6386       283
        fear     0.6591    0.7250    0.6905        80
    surprise     0.5735    0.6557    0.6119       488
     neutral     0.6915    0.6407    0.6652      1606

    accuracy                         0.7152      4968
   macro avg     0.6727    0.6762    0.6727      4968
weighted avg     0.7143    0.7152    0

In [13]:
summary_df = pd.DataFrame([
    {
        'task': 'ABSA',
        'accuracy': absa_test_metrics['accuracy'],
        'macro_f1': absa_test_metrics['macro_f1'],
    },
    {
        'task': 'Emotion',
        'accuracy': emotion_test_metrics['accuracy'],
        'macro_f1': emotion_test_metrics['macro_f1'],
    }
])

summary_df

,task,accuracy,macro_f1
0,ABSA,0.808581,0.666811
1,Emotion,0.715177,0.672748


## Random Sample Prediction Demo

In [14]:
def predict_absa(text, aspect, model, tokenizer, max_length=128):
    model.eval()
    encoded = tokenizer(
        text=str(text),
        text_pair=str(aspect),
        padding='max_length',
        truncation=True,
        max_length=max_length,
        return_tensors='pt'
    )

    input_ids = encoded['input_ids'].to(DEVICE)
    attention_mask = encoded['attention_mask'].to(DEVICE)
    token_type_ids = encoded.get('token_type_ids')
    if token_type_ids is not None:
        token_type_ids = token_type_ids.to(DEVICE)

    with torch.no_grad():
        logits = model(
            input_ids=input_ids,
            attention_mask=attention_mask,
            token_type_ids=token_type_ids,
            task='absa'
        )
        pred_id = int(torch.argmax(logits, dim=1).cpu().item())

    return ID2ABSA_LABEL[pred_id]

def predict_emotion(text, model, tokenizer, max_length=128):
    model.eval()
    encoded = tokenizer(
        text=str(text),
        padding='max_length',
        truncation=True,
        max_length=max_length,
        return_tensors='pt'
    )

    input_ids = encoded['input_ids'].to(DEVICE)
    attention_mask = encoded['attention_mask'].to(DEVICE)
    token_type_ids = encoded.get('token_type_ids')
    if token_type_ids is not None:
        token_type_ids = token_type_ids.to(DEVICE)

    with torch.no_grad():
        logits = model(
            input_ids=input_ids,
            attention_mask=attention_mask,
            token_type_ids=token_type_ids,
            task='emotion'
        )
        pred_id = int(torch.argmax(logits, dim=1).cpu().item())

    return ID2EMOTION_LABEL[pred_id]

In [ ]:
demo_samples = absa_test.sample(5).reset_index(drop=True).copy()
demo_samples['absa_pred'] = demo_samples.apply(
    lambda row: predict_absa(row['text'], row['aspect'], model, tokenizer, MAX_LENGTH),
    axis=1
)
demo_samples['emotion_pred'] = demo_samples['text'].apply(
    lambda text: predict_emotion(text, model, tokenizer, MAX_LENGTH)
)
demo_samples['absa_is_correct'] = demo_samples['absa_pred'] == demo_samples['label_name']

demo_display = demo_samples[[
    'text', 'aspect', 'label_name', 'absa_pred', 'absa_is_correct', 'emotion_pred'
]].rename(columns={
    'label_name': 'absa_gold'
})

demo_display

,text,aspect,absa_gold,absa_pred,absa_is_correct,emotion_pred
0,"The screen takes some getting use to, because it is smaller than the laptop.",screen,negative,negative,True,neutral
1,"It has plenty of memory, lots of hard drive, and great graphics.",hard drive,positive,positive,True,joy
2,"I would never wait for a table to eat, it just is not THAT great.",table,neutral,negative,False,anger
3,"The crispy chicken wasn't for us, though.",crispy chicken,negative,negative,True,neutral
4,We could only get through an appetizer and cheese fondue.,appetizer,neutral,neutral,True,neutral


Use `absa_is_correct` to compare the ABSA prediction with the SemEval gold label. For emotion, manually inspect whether the predicted emotion is reasonable for each sentence.

## Sentence-level Demo with All Aspects

In [31]:
demo_texts = absa_test['text'].drop_duplicates().sample(5).tolist()

sentence_demo_rows = []
for text in demo_texts:
    sentence_rows = absa_test[absa_test['text'] == text].copy()
    emotion_pred = predict_emotion(text, model, tokenizer, MAX_LENGTH)

    for _, row in sentence_rows.iterrows():
        absa_pred = predict_absa(row['text'], row['aspect'], model, tokenizer, MAX_LENGTH)
        sentence_demo_rows.append({
            'text': row['text'],
            'aspect': row['aspect'],
            'absa_gold': row['label_name'],
            'absa_pred': absa_pred,
            'absa_is_correct': absa_pred == row['label_name'],
            'emotion_pred': emotion_pred
        })

sentence_demo_df = pd.DataFrame(sentence_demo_rows)
sentence_demo_df

,text,aspect,absa_gold,absa_pred,absa_is_correct,emotion_pred
0,The shrimp scampi was excellent and the antipasti were plentiful.,shrimp scampi,positive,positive,True,joy
1,"The acer one computer that I bought is 17 ince screen and its hard to find lap top bags for it, but I like the big s...",screen,positive,neutral,False,joy
2,"I also love the design, the looks, the feel, and the my toshiba feature is wonderfull.",my toshiba feature,positive,positive,True,joy
3,"My GF and I still choose to eat there a lot because of diverse cocktails, the chill decor, and the decent sushi.",sushi,positive,positive,True,joy
4,"Most everything is fine with this machine: speed, capacity, build.",build,positive,positive,True,joy


In [32]:
for text in demo_texts:
    sentence_rows = sentence_demo_df[sentence_demo_df['text'] == text]
    print('=' * 120)
    print('Sentence:', text)
    print('Emotion prediction:', sentence_rows['emotion_pred'].iloc[0])
    print('-' * 120)

    for _, row in sentence_rows.iterrows():
        print(f"Aspect: {row['aspect']}")
        print(f"ABSA gold: {row['absa_gold']}")
        print(f"ABSA pred: {row['absa_pred']}")
        print(f"ABSA correct: {row['absa_is_correct']}")
        print('-' * 60)

Sentence: The shrimp scampi was excellent and the antipasti were plentiful.
Emotion prediction: joy
------------------------------------------------------------------------------------------------------------------------
Aspect: shrimp scampi
ABSA gold: positive
ABSA pred: positive
ABSA correct: True
------------------------------------------------------------
Sentence: The acer one computer that I bought is 17 ince screen and its hard to find lap top bags for it, but I like the big screen on it.
Emotion prediction: joy
------------------------------------------------------------------------------------------------------------------------
Aspect: screen
ABSA gold: positive
ABSA pred: neutral
ABSA correct: False
------------------------------------------------------------
Sentence: I also love the design, the looks, the feel, and the my toshiba feature is wonderfull.
Emotion prediction: joy
-------------------------------------------------------------------------------------------------